# FCC-ee beam-beam collision with WarpX

In this notebook, we analyze the output of a single FCC-ee Z-pole bunch
crossing simulated with WarpX. We will:

- compare the simulated horizontal beam-beam kick with the analytical model
- reconstruct passive test-particle trajectories through the collision
- visualize the bunches as they collide
- inspect beamstrahlung, radiative Bhabha, and incoherent-pair products
- estimate the luminosity and radiative Bhabha lifetime

Run the WarpX input first, then execute this notebook from the
`tutorial_1` directory so that it can find `diags/` and
`tutorial_1_utils.py`.

In [1]:
import os

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tutorial_1_utils as utils
from openpmd_viewer import OpenPMDTimeSeries, ParticleTracker
from scipy.constants import c, physical_constants
from scipy.integrate import trapezoid

classical_electron_radius = physical_constants["classical electron radius"][0]

matplotlib.rcParams.update(
    {
        "figure.figsize": (8, 5),
        "font.size": 14,
        "axes.grid": True,
        "grid.alpha": 0.25,
        "legend.frameon": False,
    }
)

## 1. FCC-ee Z-pole beam parameters

In [2]:
energy = 45.6  # beam energy [GeV]
electron_rest_energy = 0.511e-3  # [GeV]
gamma = energy / electron_rest_energy

bunch_intensity = 20.20e10  # particles per bunch
beta_x = 90e-3  # beta_x* [m]
beta_y = 0.7e-3  # beta_y* [m]
sigma_x = 7993.75e-9  # sigma_x* [m]
sigma_y = 35.20e-9  # sigma_y* [m]
sigma_z = 16.7e-3  # bunch length [m]
sigma_xprime = sigma_x / beta_x  # rms horizontal divergence [rad]
sigma_yprime = sigma_y / beta_y  # rms vertical divergence [rad]
phi = 15e-3  # half crossing angle [rad]

## 2. Bunches during the collision

WarpX records particle coordinates at several iterations. Beam 1 (blue)
moves from left to right, beam 2 (orange) moves from right to left, and the
beamstrahlung photons are shown in black. Only a subset of macroparticles is
drawn to keep these diagnostic plots responsive on a laptop.

In [3]:
# Diagnostics generated by tutorial_1_input.txt
diags_name = "diags"
n_steps = 128

# To analyze a supplied reference run instead, change both values, e.g.:
# diags_name = "diags_ref_new"
# n_steps = 512
particle_series = OpenPMDTimeSeries(os.path.join(diags_name, "particles_in"))

FileNotFoundError: [Errno 2] No such file or directory: 'diags/particles_in'

In [ ]:
beam_stride = 25
photon_stride = 5

for output_index, iteration in enumerate(particle_series.iterations):
    beam1 = utils.extract_macroparticles(
        ["beam1"], diags_name=diags_name, step=output_index
    )
    beam2 = utils.extract_macroparticles(
        ["beam2"], diags_name=diags_name, step=output_index
    )
    photons1 = utils.extract_macroparticles(
        ["pho1"], diags_name=diags_name, step=output_index
    )
    photons2 = utils.extract_macroparticles(
        ["pho2"], diags_name=diags_name, step=output_index
    )

    x1, y1, z1 = beam1[:3]
    x2, y2, z2 = beam2[:3]
    x_pho1, y_pho1, z_pho1 = photons1[:3]
    x_pho2, y_pho2, z_pho2 = photons2[:3]

    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharex=True)

    for axis, coordinate1, coordinate2, photon1, photon2, scale in (
        (axes[0], x1, x2, x_pho1, x_pho2, sigma_x),
        (axes[1], y1, y2, y_pho1, y_pho2, sigma_y),
    ):
        axis.scatter(
            z1[::beam_stride] / sigma_z,
            coordinate1[::beam_stride] / scale,
            s=4,
            alpha=0.25,
            color="tab:blue",
            linewidths=0,
            rasterized=True,
            label="beam 1",
        )
        axis.scatter(
            z2[::beam_stride] / sigma_z,
            coordinate2[::beam_stride] / scale,
            s=4,
            alpha=0.25,
            color="tab:orange",
            linewidths=0,
            rasterized=True,
            label="beam 2",
        )
        axis.scatter(
            z_pho1[::photon_stride] / sigma_z,
            photon1[::photon_stride] / scale,
            s=4,
            alpha=0.3,
            color="black",
            marker="x",
            linewidths=0.5,
            rasterized=True,
            label="beamstrahlung photons",
        )
        axis.scatter(
            z_pho2[::photon_stride] / sigma_z,
            photon2[::photon_stride] / scale,
            s=4,
            alpha=0.3,
            color="black",
            marker="x",
            linewidths=0.5,
            rasterized=True,
        )
        axis.set_xlabel(r"Longitudinal position $z/\sigma_z$")

    axes[0].set_ylabel(r"Horizontal position $x/\sigma_x^*$")
    axes[1].set_ylabel(r"Vertical position $y/\sigma_y^*$")
    axes[0].legend(loc="best", fontsize=9, markerscale=2)
    fig.suptitle(f"WarpX iteration {iteration}")
    fig.tight_layout()
    plt.show()

### Test-particle trajectories

The `test1` and `test2` species have the same initial distributions as the
electron and positron bunches, but `do_not_deposit = 1` prevents them from
contributing charge or current to the fields. They therefore act as passive
probes of the self-consistent beam-beam fields. The `trajectories` diagnostic
records these particles at every time step.

A separate `ParticleTracker` is initialized from each species at the first
iteration. Setting `preserve_particle_index=True` keeps every trajectory in
the same array column at later iterations, so successive coordinates with the
same particle ID form one path through the collision. The 3D view shows the
complete paths; the two projections make the small transverse motion easier
to read.

In [ ]:
trajectory_series = OpenPMDTimeSeries(os.path.join(diags_name, "trajectories"))
trajectory_iterations = trajectory_series.iterations

fig = plt.figure(figsize=(15, 4.5))
axis_3d = fig.add_subplot(1, 3, 1, projection="3d")
axes = [fig.add_subplot(1, 3, 2), fig.add_subplot(1, 3, 3)]

for species, label, color in (
    ("test1", "beam-1 test particles", "tab:blue"),
    ("test2", "beam-2 test particles", "tab:orange"),
):
    tracker = ParticleTracker(
        trajectory_series,
        species=species,
        iteration=trajectory_iterations[0],
        preserve_particle_index=True,
    )

    shape = (len(trajectory_iterations), tracker.N_selected)
    x_trajectories = np.empty(shape)
    y_trajectories = np.empty(shape)
    z_trajectories = np.empty(shape)

    for output_index, iteration in enumerate(trajectory_iterations):
        x, y, z = trajectory_series.get_particle(
            ["x", "y", "z"],
            species=species,
            select=tracker,
            iteration=iteration,
        )
        x_trajectories[output_index] = x
        y_trajectories[output_index] = y
        z_trajectories[output_index] = z

    for x_path, y_path, z_path in zip(
        x_trajectories.T, y_trajectories.T, z_trajectories.T
    ):
        axis_3d.plot(
            z_path / sigma_z,
            x_path / sigma_x,
            y_path / sigma_y,
            color=color,
            alpha=0.2,
            linewidth=0.8,
        )
    axes[0].plot(
        z_trajectories / sigma_z,
        x_trajectories / sigma_x,
        color=color,
        alpha=0.2,
        linewidth=0.8,
    )
    axes[1].plot(
        z_trajectories / sigma_z,
        y_trajectories / sigma_y,
        color=color,
        alpha=0.2,
        linewidth=0.8,
    )
    axis_3d.plot([], [], [], color=color, linewidth=1.5, label=label)

axis_3d.set(
    xlabel=r"$z/\sigma_z$",
    ylabel=r"$x/\sigma_x^*$",
    zlabel=r"$y/\sigma_y^*$",
    title="3D trajectories",
)
axes[0].set(
    xlabel=r"Longitudinal position $z/\sigma_z$",
    ylabel=r"Horizontal position $x/\sigma_x^*$",
    title="Horizontal trajectories",
)
axes[1].set(
    xlabel=r"Longitudinal position $z/\sigma_z$",
    ylabel=r"Vertical position $y/\sigma_y^*$",
    title="Vertical trajectories",
)
axis_3d.legend(fontsize=9)
fig.tight_layout()
plt.show()

## 3. Horizontal beam-beam kick

We begin with the incoherent (single-particle) beam-beam kick in the
horizontal plane. The kick is the change in a particle's horizontal angle
during the collision,

$$\Delta x' = x'_{\mathrm{end}} - x'_{\mathrm{start}}.$$

For Gaussian beams, it depends on the particle's horizontal offset from the
centroid of the opposing beam:

<img src="img/bb_force.png" width="480" alt="Beam-beam force versus transverse offset">

The strong hourglass effect makes the corresponding vertical comparison more
subtle for the FCC-ee Z-pole parameters, although the same reasoning applies:

<img src="img/hourglass.png" width="480" alt="FCC-ee hourglass effect">

Near the beam axis, the integrated kick is approximately linear:

$$
\Delta x'(x) \simeq -\frac{4\pi\xi_x}{\beta_x^*}x,
\qquad x\rightarrow 0.
$$

Here, $4\pi\xi_x/\beta_x^*=1/f$ is the inverse focal length of the
linearized beam-beam lens. For a crossing with half angle $\phi$,

$$
\xi_x = \frac{N_b r_e \beta_x^*}
{2\pi\gamma\sigma^*_{x,\mathrm{eff}}
(\sigma^*_{x,\mathrm{eff}} + \sigma_y^*)},
\qquad
\sigma^*_{x,\mathrm{eff}} = \sigma_x^*
\sqrt{1+\left(\frac{\sigma_z}{\sigma_x^*}\tan\phi\right)^2}.
$$

The effective horizontal size accounts for the bunch overlap at a crossing
angle:

<img src="img/tilted_beam_ellipse.png" width="480" alt="Effective beam size at a crossing angle">

In [ ]:
piwinski_angle = sigma_z / sigma_x * np.tan(phi)
sigma_x_eff = sigma_x * np.sqrt(1.0 + piwinski_angle**2)

xi_x = (
    bunch_intensity
    * classical_electron_radius
    * beta_x
    / (2.0 * np.pi * gamma * sigma_x_eff * (sigma_x_eff + sigma_y))
)
xi_y = (
    bunch_intensity
    * classical_electron_radius
    * beta_y
    / (2.0 * np.pi * gamma * sigma_y * (sigma_x_eff + sigma_y))
)

print(f"Piwinski angle: {piwinski_angle:.2f}")
print(f"Beam-beam parameters: xi_x = {xi_x:.4f}, xi_y = {xi_y:.4f}")

`openPMD-viewer` reads the same beam-1 macroparticles at the first and last
recorded iterations. Tracking matching particle IDs ensures that particles
which leave the simulation domain are not inadvertently paired with different
particles.

In [ ]:
# Select at the final iteration, then retrieve those same particles initially.
beam1_tracker = ParticleTracker(particle_series, species="beam1", iteration=n_steps)

x_start, y_start, z_start, ux_start, uy_start, uz_start = particle_series.get_particle(
    ["x", "y", "z", "ux", "uy", "uz"],
    species="beam1",
    select=beam1_tracker,
    iteration=0,
)
x_end, y_end, z_end, ux_end, uy_end, uz_end = particle_series.get_particle(
    ["x", "y", "z", "ux", "uy", "uz"],
    species="beam1",
    select=beam1_tracker,
    iteration=n_steps,
)

# For an ultrarelativistic beam, x' = p_x / p_z and y' = p_y / p_z.
xprime_start = ux_start / uz_start
yprime_start = uy_start / uz_start
xprime_end = ux_end / uz_end
yprime_end = uy_end / uz_end

The force is linear only close to the axis. Because the effective horizontal
beam size is much larger than $\sigma_x^*$ at this crossing angle, we fit the
central region $|x|<10\sigma_x^*$. Both axes are normalized so that the fitted
slope can be compared directly with $-4\pi\xi_x$.

In [ ]:
fit_limit = 10.0

x_normalized = x_start / sigma_x
kick_normalized = (xprime_end - xprime_start) / sigma_xprime
in_fit_region = np.abs(x_normalized) < fit_limit

kick_slope_warpx, kick_intercept = np.polyfit(
    x_normalized[in_fit_region],
    kick_normalized[in_fit_region],
    deg=1,
)
kick_slope_theory = -4.0 * np.pi * xi_x

print(f"Normalized kick slope from WarpX: {kick_slope_warpx:.5f}")
print(f"Analytical slope (-4 pi xi_x): {kick_slope_theory:.5f}")
print("Relative difference: " f"{abs(kick_slope_warpx / kick_slope_theory - 1.0):.1%}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.scatter(
    x_normalized,
    kick_normalized,
    s=3,
    alpha=0.25,
    color="black",
    linewidths=0,
    rasterized=True,
    label="WarpX macroparticles",
)

# Sort the abscissa so the fitted and analytical curves render as clean lines.
x_fit = np.sort(x_normalized[in_fit_region])
ax.plot(
    x_fit,
    kick_slope_warpx * x_fit + kick_intercept,
    color="tab:red",
    linewidth=2.5,
    label=rf"WarpX fit: $m={kick_slope_warpx:.4f}$",
)
ax.plot(
    x_fit,
    kick_slope_theory * x_fit + kick_intercept,
    color="tab:blue",
    linestyle="dashed",
    linewidth=2.5,
    label=rf"Theory: $-4\pi\xi_x={kick_slope_theory:.4f}$",
)

ax.axvspan(
    -fit_limit,
    fit_limit,
    color="tab:green",
    alpha=0.08,
    label=rf"fit region: $|x|<{fit_limit:g}\sigma_x^*$",
)
ax.set(
    xlabel=r"Initial horizontal position $x/\sigma_x^*$",
    ylabel=r"Normalized kick $\Delta x'/\sigma_{x'}^*$",
    title="Horizontal beam-beam kick",
)

scale_to_effective_size = sigma_x / sigma_x_eff
secondary_axis = ax.secondary_xaxis(
    "top",
    functions=(
        lambda value: value * scale_to_effective_size,
        lambda value: value / scale_to_effective_size,
    ),
)
secondary_axis.set_xlabel(r"Initial position $x/\sigma_{x,\mathrm{eff}}^*$")
ax.legend(fontsize=11, markerscale=3)
fig.tight_layout()
plt.show()

## 4. Photon spectra

The collision produces beamstrahlung photons (`pho1`, `pho2`) and radiative
Bhabha photons (`pho1_bha`, `pho2_bha`). The helper function collects both
particles still inside the domain and particles recorded at its boundaries.

In [ ]:
beamstrahlung1 = utils.extract_macroparticles(["pho1"], diags_name=diags_name)
beamstrahlung2 = utils.extract_macroparticles(["pho2"], diags_name=diags_name)
bhabha1 = utils.extract_macroparticles(["pho1_bha"], diags_name=diags_name)
bhabha2 = utils.extract_macroparticles(["pho2_bha"], diags_name=diags_name)


def photon_energy_gev(momentum_x, momentum_y, momentum_z):
    # Return photon energy [GeV] from momentum components [eV/c].
    return np.sqrt(momentum_x**2 + momentum_y**2 + momentum_z**2) * 1e-9


e_ele_bs = photon_energy_gev(*beamstrahlung1[3:6])
e_pos_bs = photon_energy_gev(*beamstrahlung2[3:6])
e_ele_bh = photon_energy_gev(*bhabha1[3:6])
e_pos_bh = photon_energy_gev(*bhabha2[3:6])

w_ele_bs = beamstrahlung1[-1]
w_pos_bs = beamstrahlung2[-1]
w_ele_bh = bhabha1[-1]
w_pos_bh = bhabha2[-1]

In [ ]:
bins = np.logspace(-12, np.log10(energy), 100)
fig, ax = plt.subplots()

ax.hist(
    e_ele_bs,
    bins=bins,
    weights=w_ele_bs,
    histtype="step",
    linewidth=1.8,
    label="from beam 1",
)
ax.hist(
    e_pos_bs,
    bins=bins,
    weights=w_pos_bs,
    histtype="step",
    linewidth=1.8,
    label="from beam 2",
)
ax.axvline(energy, linestyle=":", color="black", label="beam energy")
ax.set(
    xscale="log",
    yscale="log",
    xlabel="Photon energy [GeV]",
    ylabel="Weighted beamstrahlung photons per bin",
    title="Beamstrahlung spectrum",
)
ax.legend(fontsize=11)
fig.tight_layout()
plt.show()

The momentum acceptance is the largest relative momentum deviation for which
ring motion remains stable. If a primary beam particle emits a radiative
Bhabha photon carrying more than this fraction of the beam energy, we treat
that primary as lost before it reaches the next interaction point.

In [ ]:
momentum_acceptance = 0.01  # relative momentum acceptance (1%)

In [ ]:
bins = np.logspace(-7, np.log10(energy), 100)
fig, ax = plt.subplots()

ax.hist(
    e_ele_bh,
    bins=bins,
    weights=w_ele_bh,
    histtype="step",
    linewidth=1.8,
    label="from beam 1",
)
ax.hist(
    e_pos_bh,
    bins=bins,
    weights=w_pos_bh,
    histtype="step",
    linewidth=1.8,
    label="from beam 2",
)
ax.axvline(energy, linestyle=":", color="black", label="beam energy")
ax.axvline(
    momentum_acceptance * energy,
    linestyle="dashed",
    color="tab:red",
    label="1% momentum-acceptance threshold",
)
ax.set(
    xscale="log",
    yscale="log",
    xlabel="Photon energy [GeV]",
    ylabel="Weighted radiative Bhabha photons per bin",
    title="Radiative Bhabha photon spectrum",
)
ax.legend(fontsize=11)
fig.tight_layout()
plt.show()

## 5. Luminosity

WarpX records two complementary luminosity diagnostics in
`diags/reducedfiles/`. `DifferentialLuminosity` accumulates the luminosity
spectrum in center-of-mass energy, while `ColliderRelevant` records the
instantaneous $d\mathcal{L}/dt$. Integrating each diagnostic provides an
internal consistency check before comparison with analytical estimates.

In [ ]:
fig, ax = plt.subplots(ncols=2, nrows=1, figsize=(20, 6))

##################
# e differential #
##################

Ecom, dL_dEcom, lumi_from_spectrum = utils.get_dL_dEcom(
    os.path.join(diags_name, "reducedfiles/DiffLumi_beam1_beam2.txt")
)
print(f"Luminosity from the energy spectrum [m^-2] = {lumi_from_spectrum:.4e}")
ax[0].plot(Ecom, dL_dEcom)
ax[0].set_yscale("log")
ax[0].set(xlabel="E [eV]", ylabel=r"$d\mathcal{L}/dE$ [m$^{-2}$eV$^{-1}$]")
ax[0].set_title("Energy differential luminosity")
ax[0].axvline(2 * energy * 1e9, c="r", ls=":", label=r"$2E_{beam}$")
ax[0].legend(fontsize=24)

##################
# t differential #
##################

cr_df = pd.read_csv(
    os.path.join(diags_name, "reducedfiles/ColliderRelevant.txt"), sep=" "
)
times = cr_df["[1]time(s)"]
dL_dt = cr_df["[2]dL_dt(m^-2*s^-1)"]
ax[1].plot(times, dL_dt)
ax[1].set(xlabel="t [s]", ylabel=r"$d\mathcal{L}/dt$ [s$^{-1}$m$^{-2}$]")
ax[1].set_title("Time differential luminosity")

fig.tight_layout()

In [ ]:
lumi_warpx = trapezoid(dL_dt, times)
print(f"Luminosity from the time integral [m^-2] = {lumi_warpx:.4e}")
print(f"Energy-spectrum/time-integral ratio = {lumi_from_spectrum / lumi_warpx:.4f}")

The simplest analytical estimate includes the crossing-angle enlargement of
the horizontal beam size but neglects the hourglass effect.

In [ ]:
lumi_ip = bunch_intensity**2 / (4 * np.pi * sigma_x_eff * sigma_y)  # [m^-2] for 1 IP
lumi_ip  # [m^-2]

The numerical overlap integral in `luminosity_per_bx_hourglass` also includes
the variation of the transverse beam sizes through the collision.

In [ ]:
lumi_ip_hg = utils.luminosity_per_bx_hourglass(
    bunch_intensity, sigma_x, sigma_y, sigma_z, phi, beta_x, beta_y
)
lumi_ip_hg  # [m^-2]

In [ ]:
print(
    rf"Luminosity per bunch crossing [m^-2]: WarpX: {lumi_warpx:.4e}, formula: {lumi_ip_hg:.4e}, ratio: {lumi_warpx/lumi_ip_hg}"
)

## 6. Radiative Bhabha cross section and beam lifetime

First, estimate the cross section from WarpX. A radiative Bhabha photon above
the momentum-acceptance threshold corresponds approximately to one lost
primary beam particle.

In [ ]:
n_interaction_points = 4
ring_circumference = 90_644.838  # [m]
revolution_frequency = c / ring_circumference  # [Hz]

In [ ]:
# Weighted number of photons above the loss threshold.
loss_threshold = momentum_acceptance * energy
count_ele_lost = np.sum(w_ele_bh[e_ele_bh > loss_threshold])
count_pos_lost = np.sum(w_pos_bh[e_pos_bh > loss_threshold])

# 1 mbarn = 1e-31 m^2.
mbarn_to_m2 = 1e-31
sigma_bhabha_ele_warpx = count_ele_lost / lumi_warpx / mbarn_to_m2
sigma_bhabha_pos_warpx = count_pos_lost / lumi_warpx / mbarn_to_m2

tau_bhabha_ele_warpx = 60.0 * utils.beam_lifetime(
    sigma_bhabha_ele_warpx,
    lumi_warpx * mbarn_to_m2,
    bunch_intensity,
    n_interaction_points,
    revolution_frequency,
)
tau_bhabha_pos_warpx = 60.0 * utils.beam_lifetime(
    sigma_bhabha_pos_warpx,
    lumi_warpx * mbarn_to_m2,
    bunch_intensity,
    n_interaction_points,
    revolution_frequency,
)

Next, compute the radiative Bhabha cross section from the QED expression that
neglects the beam-size effect, and convert it into the corresponding lifetime.

In [ ]:
electron_mass_gev = 0.000511
fine_structure_constant = 1.0 / 137.036
gev_inverse_squared_to_mbarn = 0.389

cross_section_prefactor = (
    2.0
    * fine_structure_constant**3
    / electron_mass_gev**2
    * gev_inverse_squared_to_mbarn
)
qed_integral = utils.quad(
    utils.integrand_qed,
    momentum_acceptance,
    1.0,
    args=(energy, electron_mass_gev),
)[0]
sigma_bhabha_bse_off = cross_section_prefactor * qed_integral
tau_bhabha_bse_off = 60.0 * utils.beam_lifetime(
    sigma_bhabha_bse_off,
    lumi_warpx * mbarn_to_m2,
    bunch_intensity,
    n_interaction_points,
    revolution_frequency,
)

The baseline WarpX input disables the finite beam-size correction, so the
ratios below compare two calculations that both neglect that effect. To study
finite-beam-size suppression, enable
`qed_virtual_photons_do_beam_size_effect` for both beams and rerun the
simulation in a separate directory.

In [ ]:
print("Radiative Bhabha result for beam 2")
print(f"  WarpX cross section: {sigma_bhabha_pos_warpx:.2f} mbarn")
print(f"  QED cross section:   {sigma_bhabha_bse_off:.2f} mbarn")
print(f"  WarpX/QED:           {sigma_bhabha_pos_warpx / sigma_bhabha_bse_off:.3f}")
print()
print(f"  WarpX lifetime:      {tau_bhabha_pos_warpx:.2f} min")
print(f"  QED lifetime:        {tau_bhabha_bse_off:.2f} min")
print(f"  WarpX/QED:           {tau_bhabha_pos_warpx / tau_bhabha_bse_off:.3f}")

## 7. Incoherent pairs

Finally, combine the electron-positron pairs produced through the
Breit-Wheeler, Breit-Heitler, and Landau-Lifshitz processes. Before computing
the angular spectrum, rotate their momenta from the head-on simulation frame
to the physical crossing-angle frame. Their angular and spatial distributions
indicate where detector backgrounds may emerge.

In [ ]:
pair_species = ["ele_ll", "pos_ll", "ele_bh", "pos_bh", "ele_bw", "pos_bw"]
x_pair, y_pair, z_pair, px_pair, py_pair, pz_pair, w_pair = (
    utils.extract_macroparticles(pair_species, diags_name=diags_name)
)
rotation_angle = np.where(pz_pair >= 0.0, phi, -phi)
px_pair, pz_pair = (
    np.cos(rotation_angle) * px_pair + np.sin(rotation_angle) * pz_pair,
    -np.sin(rotation_angle) * px_pair + np.cos(rotation_angle) * pz_pair,
)
theta = np.arctan2(np.hypot(px_pair, py_pair), pz_pair)

theta_edges = np.linspace(0.0, np.pi, 81)
theta_counts, _ = np.histogram(theta, bins=theta_edges, weights=w_pair)
theta_widths = np.diff(theta_edges)
theta_centers = 0.5 * (theta_edges[1:] + theta_edges[:-1])

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].step(theta_centers, theta_counts / theta_widths, where="mid")
axes[0].set(
    yscale="log",
    xlabel=r"Polar angle $\theta$ [rad]",
    ylabel=r"$dN/d\theta$ [rad$^{-1}$]",
    title="Angular distribution",
)

axes[1].scatter(z_pair, x_pair, s=3, alpha=0.4, linewidths=0, rasterized=True)
axes[1].set(
    xlabel="Longitudinal position z [m]",
    ylabel="Horizontal position x [m]",
    title="Longitudinal-horizontal plane",
)

axes[2].scatter(x_pair, y_pair, s=3, alpha=0.4, linewidths=0, rasterized=True)
axes[2].set(
    xlabel="Horizontal position x [m]",
    ylabel="Vertical position y [m]",
    title="Transverse plane",
)

fig.tight_layout()
plt.show()